# GPW: Periodic Heat and Movable Stitch Lines

Select a GPU runtime, then Run all. Restart first if an older notebook
already loaded these model classes. No GitHub or separate files needed.

Edit STITCH_LEARNING_RATES in the settings cell to test one or several
values. Each value runs all four starts from scratch, with identical data
and seeds. Default 0.003 reproduces the local configuration.

Heat stays horizontal on the left and vertical on the right. Initial
stitches form a vertical, offset vertical, diagonal or horizontal line.
Each submodel starts on its own side; supports are free to cross later.
Each stitch then moves independently; there is no line constraint. Kernels are two
ARD RBFs, and uncertainty includes the full stitched residual R.

This is a posterior-adaptive likelihood-minus-KL objective, not a
fixed-model ELBO guarantee. The learned stitches need not recover the material
boundary. Heating is spatially repeating; the exterior is insulated.
Finite-element heat simulation runs on CPU; GP training uses float64 GPU.
No CPU fallback, solver retries or automatic parameter changes.

**Ablation update.** `CONDITIONS` in the settings cell lists (stitch_mode, q_rank) pairs.
`("none", 0)` uses the identical initial supports of each layout but zero stitch
constraints and no low-rank V; `("learned", 24)` reproduces the original runs.
Layout names then only describe the initial support split for the no-stitch condition.


## 1. Embedded source and dependencies


In [ ]:
import base64, hashlib, importlib, io, os, sys, zipfile
from pathlib import Path

ROOT = Path("/content/gpw_periodic_heat")
os.environ["MPLCONFIGDIR"] = str(ROOT / "results/.matplotlib")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
SOURCE_B64 = (
    "..."
)
SOURCE_ZIP = base64.b64decode(SOURCE_B64)
assert hashlib.sha256(SOURCE_ZIP).hexdigest() == "f371b60bf7f80892a237d7c09b07f26429a85b348deeef482a4a257edf210d89", "Embedded source checksum mismatch"
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(SOURCE_ZIP)) as bundle:
    for name in bundle.namelist():
        if not (ROOT / name).resolve().is_relative_to(ROOT.resolve()):
            raise ValueError("Unsafe archive member")
    bundle.extractall(ROOT)
os.chdir(ROOT)
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]
importlib.invalidate_caches()
print("Source verified:", "f371b60bf7f80892a237d7c09b07f26429a85b348deeef482a4a257edf210d89")


In [ ]:
%pip install --quiet --editable "/content/gpw_periodic_heat[heat]"


## 2. Settings


In [ ]:
import gc, json, shutil, zipfile
from dataclasses import asdict, replace
from datetime import UTC, datetime
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display
from google.colab import files
from experiments.periodic_heat_preview import PreviewConfig
from experiments.stitch_line_preview import LayoutConfig, preview
from experiments.stitch_line_heat import RunConfig, build_model, run
from pwgp.linalg import cholesky_solve

STITCH_LEARNING_RATES = [0.01]  # e.g. [0.003, 0.01, 0.03]; ignored by "none"
CONDITIONS = [("learned", 24), ("none", 0)]  # (stitch_mode, q_rank): "learned" | "fixed" | "none"
DEVICE = "cuda:0"
HEAT = PreviewConfig(
    cells=80, diffusivity=(0.6, 0.025), angle_pairs=((0, 90),),
    periods=(10, 10), baseline=1.0, amplitude=0.8, time=0.015,
    time_steps=40, supports_per_patch=124, support_margin=0.02,
    stitches=100, q_rank=24, lengthscale=0.12, outputscale=1.0,
    noise_std=0.001, weight_epsilon=0.01, seed=31,
)
LAYOUTS = LayoutConfig(
    heat=HEAT, line_centers=( (0.56, 0.5), (0.5, 0.5), (0.5, 0.5), (0.5, 0.5), (0.5, 0.5)),
    line_angles=( 90,80,60, 45, 0),
    line_names=( "shifted","80","60", "Diagonal", "Horizontal"),
    line_margin=0.05, candidate_points=10000,
)
CONFIG = RunConfig(
    layouts=LAYOUTS, observations=2000, epochs=1000, warmup_epochs=40,
    learning_rate_q=0.008, learning_rate_supports=0.01,
    learning_rate_line=0.003, learning_rate_hyperparameters=0.003,
    snapshot_every=50, prediction_batch_size=256, threads=2, device=DEVICE,
)
if not CONDITIONS or any(m not in ("learned", "fixed", "none") or int(r) != r or r < 0 for m, r in CONDITIONS):
    raise ValueError("CONDITIONS must hold (stitch_mode, nonnegative integer q_rank) pairs")
if len(set(CONDITIONS)) != len(CONDITIONS):
    raise ValueError("CONDITIONS must not repeat")
if not STITCH_LEARNING_RATES or any(not np.isfinite(v) or v <= 0 for v in STITCH_LEARNING_RATES):
    raise ValueError("STITCH_LEARNING_RATES must contain finite positive values")
if torch.device(DEVICE).type != "cuda" or not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime. This notebook does not fall back to CPU")
print({"GPU": torch.cuda.get_device_name(DEVICE), "torch": torch.__version__,
       "CUDA": torch.version.cuda, "dtype": "float64"})
RUN_ROOT = ROOT / "results/periodic_heat_colab" / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ROOT.mkdir(parents=True, exist_ok=False)
(RUN_ROOT / "source_snapshot.zip").write_bytes(SOURCE_ZIP)
PLAN = [(m, r, v) for m, r in CONDITIONS for v in (STITCH_LEARNING_RATES if m == "learned" else [None])]
display(pd.DataFrame([{"stitch_mode": m, "q_rank": r, "stitch_learning_rate": v,
                      "layouts": len(LAYOUTS.line_angles), "epochs": CONFIG.epochs,
                      "warmup": CONFIG.warmup_epochs} for m, r, v in PLAN]))


## 3. GPU checks and initial layouts


In [ ]:
for batch, size in [(2, 28), (500, 1), (1, 56)]:
    matrix = (torch.eye(size, device=DEVICE, dtype=torch.float64) + 0.01).repeat(batch, 1, 1)
    matrix.requires_grad_(True)
    rhs = torch.ones(batch, size, 1, device=DEVICE, dtype=torch.float64)
    answer = cholesky_solve(rhs, torch.linalg.cholesky(matrix))
    torch.testing.assert_close(matrix @ answer, rhs)
    answer.square().mean().backward()
    assert torch.isfinite(matrix.grad).all()
del matrix, rhs, answer
probe_heat = replace(HEAT, supports_per_patch=8, stitches=4, q_rank=2)
for mode, rank in CONDITIONS:
    probe_config = replace(CONFIG, layouts=replace(LAYOUTS, heat=probe_heat),
                           stitch_mode=mode, q_rank=min(rank, 2))
    for index in range(len(LAYOUTS.line_angles)):
        probe = build_model(probe_config, index)
        expected = 0 if mode == "none" else probe_heat.stitches
        assert probe.stitch_coefficients.shape[0] == expected and probe.q.rank == min(rank, 2)
        x = 0.05 + 0.9 * torch.rand(16, 2, device=DEVICE, dtype=torch.float64)
        loss = -probe.elbo_batch(x, torch.sin(8 * x[:, 0]), len(x))[0]
        loss.backward()
        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in probe.parameters())
        del probe, x, loss
torch.cuda.synchronize(DEVICE)
gc.collect()
torch.cuda.empty_cache()
print("GPU factorization, solves and GPW gradients passed for", CONDITIONS)
starting = preview(LAYOUTS, directory=RUN_ROOT / "initial_layouts")
display(Image(filename=str(starting / "initial_layouts.png")))


## 4. Train, visualize and save each run


In [ ]:
summaries = []
for rate_index, (mode, rank, rate) in enumerate(PLAN):
    config = replace(CONFIG, stitch_mode=mode, q_rank=rank,
                     **({} if rate is None else {"learning_rate_line": rate}))
    label = f"{mode}_rank{rank}" + ("" if rate is None else f"_lr{rate:g}")
    directory = RUN_ROOT / f"run_{rate_index}_{label}"
    display(Markdown(f"## {label}"))
    live = display(Markdown("Starting..."), display_id=True)

    def show_progress(folder, epoch, handle=live):
        handle.update(Image(filename=str(folder / "progress.png")))

    def completed(folder, metrics, rate=rate, rate_index=rate_index, label=label):
        summaries.append({"condition": label, "stitch_learning_rate": rate, **metrics})
        pd.DataFrame(summaries).to_csv(RUN_ROOT / "comparison.csv", index=False)
        display(pd.DataFrame([summaries[-1]]).T)
        parent = folder.parent
        shared = [parent / name for name in ("data.npz", "config.json", "provenance.json", "metrics.json")]
        archive = RUN_ROOT / f"run_{rate_index}_{label}_{folder.name}.zip"
        with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
            bundle.writestr("source_snapshot.zip", SOURCE_ZIP)
            for path in shared + sorted(p for p in folder.rglob("*") if p.is_file()):
                bundle.write(path, path.relative_to(parent))
        print("Saved:", archive)
        files.download(str(archive))

    result = run(config, directory=directory, capture_git=False,
                 on_snapshot=show_progress, on_complete=completed)
    for name in ("points.png", "fits.png", "training.png"):
        display(Image(filename=str(result / name)))
table = pd.DataFrame(summaries)
display(table[["condition", "layout", "stitch_count", "q_rank", "rmse_standardized",
               "nlpd_standardized", "coverage_95", "mean_stitch_displacement"]])
display(table.pivot(index="layout", columns="condition", values="rmse_standardized")
        .loc[list(LAYOUTS.line_names)])


## 5. Download the complete comparison
